# Análise dos Recém-Nascidos — Segundo Público

Duas fontes complementares:
1. **RECEM_NASCIDO (Oracle)** → dados do parto: 22.621 bebês, peso, Apgar, hora nasc.
   Vinculado ao atendimento da MÃE via ADMISSAO_CO.
2. **recem_nascido_atendimento (view)** → 31.319 atendimentos próprios dos bebês
   (internações com CID P07x, P22x etc.), vinculados via cd_atendimento_pai.

Fonte 1 é mais rica em dados clínicos do nascimento.
Fonte 2 captura os que foram internados separadamente e seus óbitos (222 casos).

Nota: no dataset principal (gestante_quadro_clinico.parquet), registros neonatais
contaminados na base materna são identificados pela coluna flag_neonatal=1
(produzida por enrich_obstetric). Os notebooks de gestantes filtram esses registros;
aqui trabalhamos diretamente com as fontes nativas de RN.

In [ ]:
from __future__ import annotations
import sys, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

warnings.filterwarnings("ignore")

# ROOT robusto: funciona independente do cwd
ROOT    = Path(__file__).resolve().parent.parent
PARQUET = ROOT / "data" / "gestante_quadro_clinico.parquet"
OUT_DIR = ROOT / "out" / f"rn_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

RN_PARQUET = ROOT / "data" / "recem_nascido_oracle.parquet"
print(f"ROOT      : {ROOT}")
print(f"PARQUET   : {PARQUET.exists()}")
print(f"RN Oracle : {RN_PARQUET}  existe={RN_PARQUET.exists()}")

if not RN_PARQUET.exists():
    print("\n⚠️  Exportando RECEM_NASCIDO do Oracle (necessário uma vez)...")
    import subprocess, sys
    subprocess.run([sys.executable, str(ROOT/"src"/"export_recem_nascido.py")], check=True)

## 1. Carga dos dados — duas fontes

In [ ]:
# ── Fonte 1: RECEM_NASCIDO do Oracle (dados ricos do parto) ─────────────────
df_rn_oracle = pd.read_parquet(str(RN_PARQUET))
df_rn_oracle.columns = df_rn_oracle.columns.str.lower()
print(f"[Fonte 1] RECEM_NASCIDO Oracle: {len(df_rn_oracle):,} bebês")

# ── Fonte 2: atendimentos próprios dos RN (internações separadas) ────────────
pq_abs = str(PARQUET.resolve())
con = duckdb.connect(str(ROOT / "data" / "gestante.duckdb"))
con.execute(f"CREATE OR REPLACE VIEW recem_nascido_atendimento AS "
            f"SELECT * FROM read_parquet('{pq_abs}') WHERE is_recem_nascido=1")
con.execute(f"CREATE OR REPLACE VIEW gestante AS "
            f"SELECT * FROM read_parquet('{pq_abs}') WHERE is_recem_nascido=0")

df_rn_atend = con.execute("""
    SELECT
        r.cd_atendimento       AS cd_atend_rn,
        r.cd_paciente          AS cd_paciente_rn,
        r.cd_atendimento_pai   AS cd_atend_mae,
        r.cd_cid               AS cid_rn,
        r.sn_obito             AS obito_rn,
        r.dt_atendimento       AS dt_internacao_rn,
        r.dt_alta              AS dt_alta_rn
    FROM recem_nascido_atendimento r
""").df()
df_rn_atend.columns = df_rn_atend.columns.str.lower()
print(f"[Fonte 2] Atendimentos RN     : {len(df_rn_atend):,}")
print(f"          Óbitos (internação) : {(df_rn_atend['obito_rn'].fillna('N')=='S').sum():,}")

# ── Dados maternos ────────────────────────────────────────────────────────────
df_mae_raw = con.execute("""
    SELECT
        cd_atendimento                       AS cd_atend_mae_key,
        cd_cid                               AS cid_mae,
        nm_paciente                          AS nm_mae,
        CAST(idade_anos AS DOUBLE)           AS idade_mae,
        ds_cid_principal                     AS ds_cid_mae,
        CAST(semanas_gestacao AS DOUBLE)     AS ig_mae_struct,
        sn_gestacao_tiss,
        sn_cesario_tiss,
        sn_complicacao_tiss,
        CAST(vl_escore_triagem AS DOUBLE)    AS escore_triagem_mae,
        CAST(qtd_evolucoes_documentadas AS DOUBLE) AS evol_mae,
        ds_queixa_principal,
        txt_sinais_sintomas,
        txt_evolucao_medica,
        txt_just_internacao,
        txt_obs_gestacao,
        txt_intercorrencias
    FROM gestante
""").df()
con.close()
df_mae_raw.columns = df_mae_raw.columns.str.lower()
print(f"[Mães]    Gestantes           : {len(df_mae_raw):,}")

# ── Merge RECEM_NASCIDO Oracle + dados maternos ───────────────────────────────
# Esta é a fusão principal: 22.621 bebês com peso/Apgar/hora + perfil da mãe
df = df_rn_oracle.merge(
    df_mae_raw[["cd_atend_mae_key","cid_mae","nm_mae","idade_mae",
                "ig_mae_struct","escore_triagem_mae","sn_cesario_tiss",
                "ds_queixa_principal","txt_sinais_sintomas","txt_evolucao_medica"]],
    left_on="cd_atend_mae",
    right_on="cd_atend_mae_key",
    how="left"
)

# Enriquece mãe com G/P/A + tipo_gestacao
from build_obstetric_features import enrich_obstetric

LLM_PAR = ROOT / "out" / "llm_extracted_20k.parquet"
df_mae_enr = enrich_obstetric(
    df_mae_raw.copy().assign(cd_atendimento=df_mae_raw["cd_atend_mae_key"]),
    llm_parquet=str(LLM_PAR) if LLM_PAR.exists() else None,
)
df_mae_enr.columns = df_mae_enr.columns.str.lower()
df_mae_enr = df_mae_enr.drop_duplicates(subset=["cd_atend_mae_key"], keep="first")

OBS_COLS = ["cd_atend_mae_key","G_obs","A_obs","C_obs","N_obs","IG_obs",
            "tipo_gestacao","perfil_historico","risco_gestacao",
            "gestacao_gemelar","gestacao_pretermo","primigesta",
            "grande_multipara_v2","tem_aborto_v2","aborto_recorrente",
            "sem_filhos_vivos","perda_previa","segmento_cid"]
OBS_COLS = [c.lower() for c in OBS_COLS if c.lower() in df_mae_enr.columns]

df = df.merge(df_mae_enr[OBS_COLS],
              left_on="cd_atend_mae", right_on="cd_atend_mae_key",
              how="left", suffixes=("","_enr"))

# Adiciona óbito da internação separada (Fonte 2) como desfecho complementar
# Agrega por cd_atend_mae: pega o caso mais grave (óbito > sobrevivente)
df_rn_atend["obito_internacao"] = (df_rn_atend["obito_rn"].fillna("N")=="S").astype(int)
df_rn_agg = (df_rn_atend.groupby("cd_atend_mae", as_index=False)
             .agg(obito_internacao=("obito_internacao","max"),
                  cid_rn_intern=("cid_rn","first"),
                  dt_internacao_rn=("dt_internacao_rn","min")))
df = df.merge(df_rn_agg, on="cd_atend_mae", how="left")
df["obito_internacao"] = df["obito_internacao"].fillna(0).astype(int)

# IG: usa a extraída do texto/LLM quando disponível, senão usa estruturada
if "ig_obs" in df.columns:
    df["ig_mae"] = df["ig_obs"].fillna(df.get("ig_semanas_mae", pd.Series(dtype=float)))
else:
    df["ig_mae"] = pd.to_numeric(df.get("ig_semanas_mae", None), errors="coerce")

df.columns = df.columns.str.lower()

# Tipos e features do RN
df["vl_peso"]         = pd.to_numeric(df["vl_peso"],         errors="coerce")
df["vl_apgar_5minuto"]= pd.to_numeric(df["vl_apgar_5minuto"],errors="coerce")
df["vl_apgar_1minuto"]= pd.to_numeric(df["vl_apgar_1minuto"],errors="coerce")
df["ig_mae"]          = pd.to_numeric(df["ig_mae"],           errors="coerce")
df["hr_nascimento"]   = pd.to_datetime(df["hr_nascimento"],   errors="coerce")

# Renomeia para compatibilidade com o resto do notebook
df = df.rename(columns={
    "vl_peso":          "peso_g",
    "vl_apgar_1minuto": "apgar_1min",
    "vl_apgar_5minuto": "apgar_5min",
    "tp_obito_neonatal": "obito_parto_flag",
})

df["obito_rn"]        = (df["obito_parto_flag"].notna() | df["obito_internacao"]==1).astype(int)
df["vinculado_mae"]   = df["cd_atend_mae"].notna().astype(int)
df["baixo_peso"]      = (df["peso_g"] < 2500).astype(float)
df["muito_baixo"]     = (df["peso_g"] < 1500).astype(float)
df["micro"]           = (df["peso_g"] < 1000).astype(float)
df["macrossomia"]     = (df["peso_g"] >= 4000).astype(float)
df["apgar_dep"]       = (df["apgar_5min"] < 7).astype(float)
df["prematuro"]       = (df["ig_mae"] < 37).astype(float)
df["mt_prematuro"]    = (df["ig_mae"] < 32).astype(float)
df["cesario"]         = (df["sn_cesario_tiss"].fillna("N")=="S").astype(float)
df["nascido_vivo"]    = df["cd_declaracao_nascido_vivo"].notna().astype(int)

# Segmento CID materno
def _seg(c):
    if pd.isna(c): return "SEM_CID"
    c = str(c).upper()
    if c[:3] in [f"O{i:02d}" for i in range(10,17)]: return "HIPERTENSIVOS"
    if c[:3]=="O24": return "DIABETES_GEST"
    if c[:3] in ["O60","O42","O47"]: return "PARTO_PREMATURO"
    if c[0]=="O" and c[1] in "234": return "HEMORRAGIA"
    if c[:3] in ["O98","O99"]: return "COMORBIDADES"
    if c[0]=="O": return "OUTROS_OBST"
    return "OUTROS"
df["segmento_mae"] = df["cid_mae"].apply(_seg)

df["faixa_peso"] = pd.cut(
    df["peso_g"],
    bins=[0,1000,1500,2500,4000,10000],
    labels=["<1kg","1–1.5kg","1.5–2.5kg","2.5–4kg",">4kg"],
    right=False
)

print(f"\nDataset final (Fonte 1 + enriquecimento materno):")
print(f"  Total bebês               : {len(df):,}")
print(f"  Com peso                  : {df['peso_g'].notna().sum():,} ({df['peso_g'].notna().mean()*100:.1f}%)")
print(f"  Com Apgar 5min            : {df['apgar_5min'].notna().sum():,}")
print(f"  Com IG materna            : {df['ig_mae'].notna().sum():,}")
print(f"  Baixo peso (<2.500g)      : {df['baixo_peso'].sum():,.0f} ({df['baixo_peso'].mean()*100:.1f}%)")
print(f"  Apgar deprimido (<7)      : {df['apgar_dep'].sum():,.0f} ({df['apgar_dep'].mean()*100:.1f}%)")
print(f"  Óbito no parto            : {(df['obito_parto_flag'].notna()).sum():,}")
print(f"  Óbito na internação       : {df['obito_internacao'].sum():,}")
print(f"  Óbito combinado           : {df['obito_rn'].sum():,}")
print(f"  Com perfil G/A/N materno  : {df['perfil_historico'].notna().sum() if 'perfil_historico' in df.columns else 0:,}")

df_mae = con.execute("""
    SELECT
        cd_atendimento                       AS cd_atend_mae_key,
        cd_cid                               AS cid_mae,
        nm_paciente                          AS nm_mae,
        CAST(idade_anos AS DOUBLE)           AS idade_mae,
        ds_cid_principal                     AS ds_cid_mae,
        CAST(semanas_gestacao AS DOUBLE)     AS ig_mae,
        sn_gestacao_tiss,
        sn_cesario_tiss,
        sn_complicacao_tiss,
        CAST(vl_escore_triagem AS DOUBLE)    AS escore_triagem_mae,
        CAST(qtd_evolucoes_documentadas AS DOUBLE) AS evol_mae,
        ds_queixa_principal,
        txt_sinais_sintomas,
        txt_evolucao_medica,
        txt_just_internacao,
        txt_obs_gestacao,
        txt_intercorrencias
    FROM gestante
""").df()

# (bloco de carga antigo removido — conexão já foi fechada acima)

## 4. Distribuições Gerais

In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(20, 11),
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1.05, 1.05, 1.55]}
)
axes = axes.flatten()

# Peso
df["peso_g"].dropna().hist(ax=axes[0], bins=40, color="steelblue", edgecolor="white")
for v,c,l in [(2500,"red","2.500g"),(1500,"orange","1.500g"),(4000,"green","4.000g")]:
    axes[0].axvline(v, color=c, ls="--", lw=1.5, label=l)
axes[0].set_title("Peso ao Nascer (g)"); 
axes[0].legend(fontsize=8); 
axes[0].grid(alpha=0.3)

# Apgar 5min
df["apgar_5min"].dropna().hist(ax=axes[1], bins=10, color="seagreen", edgecolor="white")
axes[1].axvline(7, color="red", ls="--", lw=1.5, label="<7 deprimido")
axes[1].set_title("Apgar 5 min"); 
axes[1].legend(fontsize=8); 
axes[1].grid(alpha=0.3)

# Top CIDs
CID_MAP = {"P073":"Prematuridade extrema","P221":"Angústia resp.","P229":"Desconforto resp.",
           "P228":"Taquipneia transit.","P589":"Icterícia","O809":"Parto normal",
           "Z039":"Obs. s/diagnóstico","P220":"SRDS tipo I"}
tc = df["cid_rn"].value_counts().dropna().head(7)
tc.index = [CID_MAP.get(c,c) for c in tc.index]
axes[2].barh(tc.index[::-1], tc.values[::-1], color="steelblue")
axes[2].set_title("Top CIDs Neonatais"); 
axes[2].grid(axis="x",alpha=0.3)

# Óbito por faixa de peso
fp = df.groupby("faixa_peso",observed=True)["obito_rn"].agg(["mean","count"]).reset_index()
fp["mean"] *= 100
cors = ["#d62728" if v>20 else "#ff7f0e" if v>5 else "#2ca02c" for v in fp["mean"]]
axes[3].barh(fp["faixa_peso"].astype(str)[::-1], fp["mean"][::-1], color=cors[::-1])
axes[3].set_xlabel("% óbito"); 
axes[3].set_title("Óbito por Faixa de Peso"); 
axes[3].grid(axis="x",alpha=0.3)

for i,(_, row) in enumerate(fp[::-1].iterrows()):
    axes[3].text(row["mean"]+0.2, i, f"n={int(row['count'])}", va="center", fontsize=8)

# Óbito por segmento materno
if "segmento_mae" in df.columns:
    seg = (df.groupby("segmento_mae")["obito_rn"]
             .agg(["mean","count"]).reset_index()
             .query("count >= 10")
             .sort_values("mean", ascending=False))
    seg["mean"] *= 100
    cs2 = ["#d62728" if v>2 else "#ff7f0e" if v>0.5 else "#2ca02c" for v in seg["mean"]]
    axes[4].barh(seg["segmento_mae"][::-1], seg["mean"][::-1], color=cs2[::-1])
    axes[4].set_xlabel("% óbito neonatal"); axes[4].set_title("Óbito por CID Materno")
    axes[4].grid(axis="x",alpha=0.3)

# Peso por IG materna
if df["ig_mae"].notna().sum() > 50:
    df["ig_grp"] = pd.cut(df["ig_mae"],bins=[0,28,32,34,37,40,50],
                           labels=["<28s","28-31s","32-33s","34-36s","37-40s",">40s"],right=False)
    ig_p = df.groupby("ig_grp",observed=True)["peso_g"].mean().dropna()
    axes[5].bar(ig_p.index.astype(str), ig_p.values, color="steelblue")
    axes[5].axhline(2500, color="red", ls="--", lw=1, label="2.500g")
    axes[5].set_title("Peso Médio por IG Materna"); axes[5].legend(fontsize=8)
    axes[5].grid(axis="y",alpha=0.3)
    plt.setp(axes[5].xaxis.get_majorticklabels(), rotation=30)
else:
    axes[5].text(0.5,0.5,"IG insuficiente",ha="center",va="center",
                 transform=axes[5].transAxes,fontsize=12)

plt.suptitle("Perfil dos Recém-Nascidos", fontsize=14, y=1.01)
#plt.tight_layout()
plt.savefig(PLOT_DIR/"01_perfil_geral_rn.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close()

## 5. Exemplo 1 — Perfil G/A/N Materno × Óbito Neonatal

In [ ]:
print("="*60)
print("EXEMPLO 1 — Perfil Histórico Materno × Óbito Neonatal")
print("="*60)

if "perfil_historico" in df.columns and df["perfil_historico"].notna().sum() > 50:
    # Sintaxe correta: agg com lista de strings no SeriesGroupBy
    perfil_ob = (
        df.groupby("perfil_historico")["obito_rn"]
          .agg(["count","sum","mean"])
          .rename(columns={"count":"n","sum":"obitos","mean":"taxa"})
          .reset_index()
          .query("n >= 10")
          .sort_values("taxa", ascending=False)
    )
    perfil_ob["taxa_pct"] = (perfil_ob["taxa"]*100).round(2)

    ref = perfil_ob[perfil_ob["perfil_historico"]=="PRIMIGESTA"]["taxa"].values
    if len(ref) and ref[0] > 0:
        perfil_ob["RR_primigesta"] = (perfil_ob["taxa"]/ref[0]).round(2)
    else:
        perfil_ob["RR_primigesta"] = np.nan

    print(perfil_ob[["perfil_historico","n","obitos","taxa_pct","RR_primigesta"]].to_string(index=False))

    fig, ax = plt.subplots(figsize=(9,5))
    cores_p = ["#d62728" if v>1 else "#ff7f0e" if v>0.3 else "#2ca02c"
               for v in perfil_ob["taxa_pct"]]
    ax.barh(perfil_ob["perfil_historico"][::-1], perfil_ob["taxa_pct"][::-1],
            color=cores_p[::-1])
    for i,(_, row) in enumerate(perfil_ob[::-1].reset_index(drop=True).iterrows()):
        rr = f"RR={row['RR_primigesta']:.1f}×" if not pd.isna(row["RR_primigesta"]) else ""
        ax.text(row["taxa_pct"]+0.02, i, f"n={int(row['n'])} {rr}",
                va="center", fontsize=8)
    ax.set_xlabel("Taxa de Óbito Neonatal (%)")
    ax.set_title("Óbito Neonatal por Perfil Histórico Materno (G/A/N)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR/"02_obito_por_perfil_materno.png", dpi=150)
    plt.show(); plt.close()
else:
    print("  Perfil histórico insuficiente — expanda extração LLM/regex.")

## 6. Exemplo 2 — Mãe com PE/HAS × Desfecho do RN

In [ ]:
print("\n"+"="*60)
print("EXEMPLO 2 — Mãe Hipertensiva (O10–O16) × Desfecho RN")
print("="*60)

for nome, mask in [
    ("Com PE/HAS (O10–O16)", df["cid_mae"].fillna("").str.match(r"^O1[0-6]")),
    ("Sem PE/HAS",           ~df["cid_mae"].fillna("").str.match(r"^O1[0-6]") & df["cid_mae"].notna()),
]:
    sub = df[mask]
    if len(sub) == 0: continue
    print(f"\n  {nome} (n={len(sub):,}):")
    print(f"    Óbito neonatal : {sub['obito_rn'].sum():,} ({sub['obito_rn'].mean()*100:.2f}%)")
    print(f"    Baixo peso     : {sub['baixo_peso'].sum():,.0f} ({sub['baixo_peso'].mean()*100:.1f}%)")
    if sub["peso_g"].notna().sum() > 5:
        print(f"    Peso médio     : {sub['peso_g'].mean():.0f}g")
    if sub["apgar_5min"].notna().sum() > 5:
        print(f"    Apgar 5min     : {sub['apgar_5min'].mean():.1f}  deprimido: {sub['apgar_dep'].sum():,.0f}")

## 7. Exemplo 3 — Heatmap IG × Peso × Óbito

In [ ]:
print("\n"+"="*60)
print("EXEMPLO 3 — Heatmap: IG Materna × Peso ao Nascer × Óbito")
print("="*60)

df_hm = df[df["ig_mae"].notna() & df["peso_g"].notna()].copy()
if len(df_hm) > 50:
    df_hm["cat_ig"]   = pd.cut(df_hm["ig_mae"],bins=[0,32,34,37,40,50],
                                labels=["<32s","32-33s","34-36s","37-40s",">40s"],right=False)
    df_hm["cat_peso"] = pd.cut(df_hm["peso_g"],bins=[0,1500,2500,4000,10000],
                                labels=["<1.5kg","1.5-2.5kg","2.5-4kg",">4kg"],right=False)

    pivot = (df_hm.groupby(["cat_ig","cat_peso"],observed=True)["obito_rn"]
                  .agg(["mean","count"])
                  .unstack("cat_peso")["mean"] * 100)

    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(9,5))
        sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
                    linewidths=0.5, ax=ax, annot_kws={"size":10})
        ax.set_title("Taxa de Óbito Neonatal (%) — IG Materna × Peso ao Nascer")
        ax.set_xlabel("Peso ao Nascer"); ax.set_ylabel("IG Materna")
        plt.tight_layout()
        plt.savefig(PLOT_DIR/"03_heatmap_ig_peso.png", dpi=150)
        plt.show(); plt.close()
        print("  Heatmap gerado.")
    else:
        print("  Dados insuficientes.")
else:
    print("  Amostra insuficiente para o heatmap.")

## 8. Estudos Causais (AIPW)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import norm as _norm

def _pre(X):
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat = [c for c in X.columns if c not in num]
    return ColumnTransformer([
        ("num", Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler())]), num),
        ("cat", Pipeline([("i",SimpleImputer(strategy="most_frequent")),
                          ("o",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), cat),
    ], remainder="drop")

def _aipw(df_a, T, Y, conf, n_boot=300):
    CONF = [c for c in conf if c in df_a.columns]
    sub  = df_a[[T,Y]+CONF].dropna(subset=[T,Y])
    sub  = sub[sub[Y].isin([0,1])]
    if sub[T].sum() < 20 or (sub[T]==0).sum() < 20:
        return None
    Tv, Yv = sub[T].values.astype(int), sub[Y].values.astype(int)
    pre = _pre(sub[CONF]); X = pre.fit_transform(sub[CONF])
    ps  = np.clip(GradientBoostingClassifier(n_estimators=200,max_depth=3,learning_rate=0.05,
                  random_state=42).fit(X,Tv).predict_proba(X)[:,1], 0.05, 0.95)
    Xo  = np.hstack([X,Tv.reshape(-1,1)])
    om  = GradientBoostingClassifier(n_estimators=200,max_depth=3,learning_rate=0.05,random_state=42).fit(Xo,Yv)
    mu1 = om.predict_proba(np.hstack([X,np.ones((len(X),1))]))[:,1]
    mu0 = om.predict_proba(np.hstack([X,np.zeros((len(X),1))]))[:,1]
    s1  = mu1 + Tv*(Yv-mu1)/(ps+1e-9)
    s0  = mu0 + (1-Tv)*(Yv-mu0)/(1-ps+1e-9)
    ate = (s1-s0).mean()
    rng = np.random.default_rng(42)
    b   = [(s1[rng.integers(0,len(Yv),len(Yv))]-s0[rng.integers(0,len(Yv),len(Yv))]).mean()
           for _ in range(n_boot)]
    se  = np.std(b); lo,hi = np.percentile(b,[2.5,97.5])
    pval= 2*(1-_norm.cdf(abs(ate/se))) if se>0 else 1.0
    return {"n":len(sub),"n_exp":int(Tv.sum()),
            "prev_ctrl":round(Yv[Tv==0].mean()*100,2),
            "prev_exp": round(Yv[Tv==1].mean()*100,2),
            "ate_pp":round(ate*100,4),
            "ic_lo":round(lo*100,4),"ic_hi":round(hi*100,4),"p":round(pval,4)}

CONF = [c for c in ["idade_mae","segmento_mae","cesario","escore_triagem_mae",
                     "G_obs","A_obs","risco_gestacao","gestacao_gemelar"]
        if c in df.columns]

# Flag PE para C4
df["T_pe"] = df["cid_mae"].fillna("").str.match(r"^O1[0-6]").astype(int)

estudos = []
print("="*60); print("ESTUDOS CAUSAIS — RN"); print("="*60)

for titulo, T_col, desc in [
    ("C1 — Prematuridade <37s → Óbito",     "prematuro",  "prematuro vs a termo"),
    ("C2 — Muito Prematuro <32s → Óbito",   "mt_prematuro","<32s vs ≥32s"),
    ("C3 — Baixo Peso <2.500g → Óbito",     "baixo_peso", "baixo peso vs adequado"),
    ("C4 — Mãe PE/HAS → Óbito RN",          "T_pe",       "mãe O10–O16 vs outras"),
]:
    dc = df[df["vinculado_mae"]==1].copy()
    r  = _aipw(dc, T_col, "obito_rn", CONF)
    if r is None:
        print(f"\n  {titulo}: SKIP (amostra insuficiente)")
        continue
    estudos.append({"estudo":titulo,**r})
    sig = "***" if r["p"]<0.001 else "**" if r["p"]<0.01 else "*" if r["p"]<0.05 else "ns"
    print(f"\n  {titulo}  [{desc}]")
    print(f"  N={r['n']:,}  T=1:{r['n_exp']:,}  ctrl:{r['prev_ctrl']}%  exp:{r['prev_exp']}%")
    print(f"  ATE={r['ate_pp']:+.2f}pp  IC95%[{r['ic_lo']:+.2f},{r['ic_hi']:+.2f}]  p={r['p']:.4f} {sig}")

## 9. Forest Plot

In [ ]:
if estudos:
    est = pd.DataFrame(estudos)
    fig, ax = plt.subplots(figsize=(10, max(4, len(est)*1.3)))
    for i, row in est.iterrows():
        cor = "#d62728" if row["ate_pp"]>0 else "#2ca02c"
        ax.errorbar(row["ate_pp"], i,
                    xerr=[[row["ate_pp"]-row["ic_lo"]],[row["ic_hi"]-row["ate_pp"]]],
                    fmt="o", color=cor, capsize=5, markersize=8)
        sig = "***" if row["p"]<0.001 else "**" if row["p"]<0.01 else "*" if row["p"]<0.05 else "ns"
        ax.text(row["ic_hi"]+0.2, i, f"p={row['p']:.3f} {sig}", va="center", fontsize=9)
    ax.axvline(0, color="black", lw=0.8, ls="--")
    ax.set_yticks(range(len(est)))
    ax.set_yticklabels(est["estudo"], fontsize=9)
    ax.set_xlabel("ATE (pp — óbito neonatal)")
    ax.set_title("Forest Plot — Fatores de Risco para Óbito Neonatal (AIPW)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR/"04_forest_rn.png", dpi=150)
    plt.show(); plt.close()

## 10. Tabela 1 dos RN

In [ ]:
def _pct(s):
    v = s.fillna(0)
    return f"{int(v.sum()):,} ({v.mean()*100:.1f}%)"
def _miq(s):
    s = s.dropna()
    if len(s)==0: return "—"
    q1,q3=s.quantile([.25,.75])
    return f"{s.median():.1f} [{q1:.1f}–{q3:.1f}]"

t1 = {
    "N total RN":                   f"{len(df):,}",
    "Vinculados à mãe":             _pct(df["vinculado_mae"].astype(float)),
    "── Desfechos ──":              "",
    "  Óbito neonatal":             _pct(df["obito_rn"].astype(float)),
    "  Apgar 5min < 7":             _pct(df["apgar_dep"]),
    "  Baixo peso (<2.500g)":       _pct(df["baixo_peso"]),
    "  Muito baixo peso (<1.500g)": _pct(df["muito_baixo"]),
    "  Macrossomia (>4.000g)":      _pct(df["macrossomia"]),
    "── Biometria ──":              "",
    "  Peso (g)":                   _miq(df["peso_g"]),
    "  Apgar 1min":                 _miq(df["apgar_1min"]),
    "  Apgar 5min":                 _miq(df["apgar_5min"]),
    "── Prematuridade ──":          "",
    "  Prematuro (<37s)":           _pct(df["prematuro"]),
    "  Muito prematuro (<32s)":     _pct(df["mt_prematuro"]),
    "── Perfil materno ──":         "",
    "  Mãe hipertensiva":           _pct(df["T_pe"].astype(float)),
}
# Adiciona colunas do enriquecimento se disponíveis
for col, lbl in [("tem_aborto_v2","Mãe c/ aborto ant. (A≥1)"),
                  ("grande_multipara_v2","Mãe grande multípara (G≥5)"),
                  ("gestacao_gemelar","Gestação gemelar")]:
    if col in df.columns:
        t1[f"  {lbl}"] = _pct(pd.to_numeric(df[col],errors="coerce").fillna(0).astype(float))

tab1 = pd.DataFrame(list(t1.items()), columns=["Característica","Valor"])
print("\nTabela 1 — Recém-Nascidos:")
print(tab1.to_string(index=False))
tab1.to_csv(OUT_DIR/"tabela1_rn.csv", index=False)
if estudos:
    pd.DataFrame(estudos).to_csv(OUT_DIR/"estudos_causais_rn.csv", index=False)

print(f"\n✅ Concluído — {OUT_DIR}")
print(f"   tabela1_rn.csv | estudos_causais_rn.csv | plots/ (4 figuras)")